# Learning of filter

## Training data
- lots of noised sinograms and original images

## loss function
- MSE between reconstructed image and original image

## learning paramaters
- We are training a noise-supression/windowing function that is multiplied by the standard ramp filter during back projection.
- start with a latent grid of ~400 (mirrored) gaussians, of varying mean and variance,
- paramaters are the 400 weights of each Gaussian. 
- Allow negative weights. 
- (Consider regularization or filtering of some sort, since the gaussians will be highly overlapping. So that the learning doesn't have too many degrees of freedom to where it gets lost and is unable to choose a direction to optimize toward.)

## Learning architecture
- gradient decsent and **(one-step)** backprop 
- compute the gradient of the loss function with respect to each paramater,
- ~~backrpopogate according to ADAM or some other gradient descent procedure.~~
- Since everything is linear, "one-step backprpop using a MSE loss function" is just multi-linear regression. 
- ~~(could nievergelt's elemtary inversion technique be helpful in allowing us to take gradients of the filtered back projection?)~~ 

# Filter Learning Ideas

Ok, to develop a little more my inclinations here, we want to learn a filter which is a gaussin mixture, by using gradient descent backprop. 

Training set would be paired oringal images and noised sinograms. ~~Can use online datasets which already include paired sinograms and reconstructions (so assuming they're accurate and treating the CT image as if it's the original image.) Can also generate our own bitmap images, or use any random bitmap image.~~ Generate our own ellipse phantoms. Then, can use a number of noise generators with different paramaters. So our data set is very easily expandable. (can use the same image with multiple different layered noise treatments.)

Can implement our own ~~forward radon transform and~~ filtered back projection, so that we can choose the filter, and also so that we can ensure the transforms are differentiable. 

Noising can be informed by actual CT sensor noise. Can do training runs on differnt data sets of mixed or only one type of noise, and differing magnitudes of noise. Then can evaluate learned filters performance on different kinds of noise.

We want to develop a filter that is good at surpressing noise, as well as accurately doing FBP without artifacts. 

Bell- shaped filters are good at the first, ramp-shaped filters are good at the second. What is good at both? 

Include the standard ramp filter, and then multiply by a noise-supressing filter window on top of that: $H(\omega) = |\omega| W(\omega)$, where $W(\omega)$ is the mixture.

Assume the goal filter window is ~~a gaussin mixture.~~ (Or rather, a "difference of gaussians," so that negative weights are allowed.) ~~Furthermore, can limit number of gaussians, to say 6. Where 3 of them have man 0, and the other 3 are mirrored accross 0. Paramters are 6 variances, 3 non-zero means, and 6 weights. (Not sure how we could possibly implement a capacity to add more filters as a trainable parameter.)~~

Train $W(\omega)$ by initializing a large finite 2d grid of gaussians of varying means and variances. Then the paramaters can be the weight of each gaussian. (and allow negative weights.)

Perform backprop (1 step) and gradient descent, utilizing different subsets of the training data. 

Can use various standard filters (or rather, gaussian mixture approximations therof) as starting points and see if the learned filters converge.

The likelihood that we find some undiscovered effective filter shape is probably pretty low; rather, the main goal of this exercise would be to demonstrate that our learning procedure converges to one/multiple of the commonly used filters.


# Next step: filter neural network

So in the above, our approach is to initialize a filter 

\begin{align*} H(\omega) = |\omega| W(\theta; \omega) \end{align*}

where $W(\omega)$ is the learnable gaussian mixture. This filter applies the same way to any input sinogram.

Next step: replace the gaussian mixture function with some neural network, say a MLP:

\begin{align*} H(\omega) = |\omega| R(\theta; \omega, S) \end{align*}

which can also learn to modulate itself based on the properties of the sinogram as a whole. 

~~The input layer will then not just be the individual input sinograms,~~

Or rather, could we use a CNN? So that the input projections are not just considered individually, but also in groups, so that we can get something that is responsive to large-scale features in the sinogram? 

- (Another question, could we make the filter depend on the projection angle as well, so that it can learn to identify and deal with angle-dependent noise? Does this require an expanded input layer, or can this also be achieved by using a CNN?) 

# Application with regard to reconstruction of probability distributions from marginal densities

- (Can we derive some sort of approximate to cramer-wold which guaruntees our construction is within a certain bound, dependent on the number (and arrangement) of projections?)

Next we want to turn our sights away from medical imaging CT and toward reconstruction of probability distributions from marginal data. 

Can we use the same procedure (data syntesis and then macine learning) to train a filter window which suppresses statistical noise? 

Is the neural network ramp filter we trained on medical imaging data effective at filtering statistical noise out-of-the-box? Conversely, if we train a neural network only on probability distribution data, can it effectively filter CT noise? 